# IMDb Movie Rating Classification (2003–2022)

This notebook compares three multiclass classifiers and two binary classifiers using IMDb movie metadata.

**Dataset:** Kaggle — Top 100 Popular Movies from 2003 to 2022 (IMDb).  
**Target:** IMDb rating bands (not a direct popularity label).

Download `movies.csv` from Kaggle and place it in `../data/movies.csv` before running.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42
DATA_PATH = Path("../data/movies.csv")


## 1. Load and inspect the dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Download the Kaggle CSV and put it in the data folder."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
df.info()


## 2. Clean data and prepare features

In [ ]:
def numeric_from_text(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    match = re.search(r"-?\d+(?:\.\d+)?", str(value).replace(",", ""))
    return float(match.group()) if match else np.nan

df = df.drop_duplicates().copy()
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df[df["Rating"].between(0, 10)].copy()

for col in ["Year", "Month", "Runtime", "Budget", "Income"]:
    df[col] = df[col].apply(numeric_from_text)

feature_cols = [
    "Year", "Month", "Certificate", "Runtime", "Genre",
    "Budget", "Income", "Country_of_origin"
]
X = df[feature_cols].copy()
rating = df["Rating"]

print("Cleaned shape:", df.shape)
print("Missing values in predictors:")
display(X.isna().sum().to_frame("Missing values"))


## 3. Define classification targets

**Multiclass:** Poor (<5), Average (5 to <7), Good (7 to <8.5), Excellent (8.5 to 10).  
**Binary:** Low-rated (<7), High-rated (≥7).


In [ ]:
multiclass_y = pd.cut(
    rating,
    bins=[-np.inf, 5, 7, 8.5, np.inf],
    labels=["Poor", "Average", "Good", "Excellent"],
    right=False
)
binary_y = pd.Series(
    np.where(rating < 7, "Low-rated", "High-rated"),
    index=df.index,
    name="Rating_Binary"
)

display(multiclass_y.value_counts(dropna=False).to_frame("Count"))
display(binary_y.value_counts().to_frame("Count"))


## 4. Preprocessing pipeline

In [ ]:
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)


## 5. Model evaluation function

In [ ]:
def evaluate_models(X_data, y_data, task_name, model_list):
    valid = y_data.notna()
    X_task = X_data.loc[valid]
    y_task = y_data.loc[valid].astype(str)

    X_train, X_test, y_train, y_test = train_test_split(
        X_task, y_task,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_task
    )

    rows = []
    fitted_pipelines = {}
    for model_name, estimator in model_list:
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", estimator)
        ])
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)

        rows.append({
            "Model": model_name,
            "Classification Type": task_name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(y_test, predictions, average="weighted", zero_division=0),
            "Recall": recall_score(y_test, predictions, average="weighted", zero_division=0),
            "F1 Score": f1_score(y_test, predictions, average="weighted", zero_division=0)
        })
        fitted_pipelines[model_name] = pipeline
        print(f"\\n{task_name} — {model_name}")
        print(classification_report(y_test, predictions, zero_division=0))

    return pd.DataFrame(rows), fitted_pipelines


## 6. Multiclass classification

In [ ]:
multiclass_models = [
    ("Logistic Regression", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ("Decision Tree", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced")),
    ("Random Forest", RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced"
    ))
]

multiclass_results, multiclass_fitted = evaluate_models(
    X, multiclass_y, "Multiclass", multiclass_models
)
display(multiclass_results)


## 7. Binary classification

In [ ]:
binary_models = [
    ("KNN", KNeighborsClassifier(n_neighbors=5)),
    ("SVM", SVC(kernel="rbf", class_weight="balanced"))
]

binary_results, binary_fitted = evaluate_models(
    X, binary_y, "Binary", binary_models
)
display(binary_results)


## 8. Combined comparison

In [ ]:
all_results = pd.concat([multiclass_results, binary_results], ignore_index=True)
display(all_results.round(3))

results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)
all_results.to_csv(results_dir / "model_comparison_recomputed.csv", index=False)

ax = all_results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score"]].plot(
    kind="bar", figsize=(12, 6), ylim=(0, 1),
    title="Performance Comparison of Classification Models"
)
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(results_dir / "model_comparison_recomputed.png", dpi=180)
plt.show()


## 9. Interpretation

- Compare models **within the same task** (multiclass with multiclass; binary with binary).
- Random Forest was the best multiclass model in the previously supplied results.
- SVM was the best binary model in the previously supplied results.
- The two tasks have different target structures, so their accuracy values are not a fair direct head-to-head comparison.
- The notebook's newly computed scores may differ from the previously supplied table. Use the newly computed results for a final reproducible submission.
